# 🛒 Olist Brazilian E-Commerce — Funnel Analysis

> A full funnel analysis of 99,441 real orders across a Brazilian marketplace (Sep 2016–Oct 2018).
> Four funnel frameworks + cohort retention + RFM segmentation.

---

## 📋 Table of Contents
1. [Executive Summary](#1.-Executive-Summary)
2. [Business Problem](#2.-Business-Problem)
3. [Methodology](#3.-Methodology)
4. [Skills & Technologies](#4.-Skills-&-Technologies)
5. [Data Loading & Setup](#5.-Data-Loading-&-Setup)
6. [Funnel 1 — Order Lifecycle](#6.-Funnel-1-—-Order-Lifecycle)
7. [Funnel 2 — Revenue Conversion](#7.-Funnel-2-—-Revenue-Conversion)
8. [Funnel 3 — Delivery Performance](#8.-Funnel-3-—-Delivery-Performance)
9. [Funnel 4 — Customer Satisfaction](#9.-Funnel-4-—-Customer-Satisfaction)
10. [Cohort Retention](#10.-Cohort-Retention)
11. [Drop-off Deep Dive](#11.-Drop-off-Deep-Dive)
12. [RFM Segmentation](#12.-RFM-Segmentation)
13. [Results & Business Recommendations](#13.-Results-&-Business-Recommendations)
14. [Next Steps](#14.-Next-Steps)

---
## 1. Executive Summary

This analysis applies **four funnel frameworks** to the Olist Brazilian E-Commerce dataset — a real marketplace transaction log with **99,441 orders**, **96,096 unique customers**, and **73 product categories** spanning Sep 2016 to Oct 2018.

| Funnel | Question Answered |
|---|---|
| **Order Lifecycle** | Of all orders placed, how many complete each stage through to a 5-star review? |
| **Revenue Conversion** | Of gross order value billed, how much revenue is actually delivered and collected? |
| **Delivery Performance** | How fast and how reliably are orders delivered? Where does timing break down? |
| **Customer Satisfaction** | Of delivered orders, how many customers review and what do they say? |

### 🔑 Headline Findings
- **97.0%** of orders are successfully delivered — excellent fulfilment rate
- **91.9%** of deliveries arrive on-time (before estimated delivery date)
- **76.3%** of reviewers leave a positive rating (4–5 stars)
- Median end-to-end delivery time is **10 days**
- **Credit card** dominates at ~74% of payment volume
- **23,560 customers** are classified as At Risk or Lost in RFM — recoverable churn opportunity

---
## 2. Business Problem

Olist operates a marketplace connecting Brazilian SME sellers to customers. Key analytical challenges include:

- **Where in the order journey do customers drop off?** — Understanding which stages lose the most volume
- **What's the true revenue picture?** — After cancellations and fulfilment failures, what actually reaches customers?
- **Is delivery performance consistent?** — Delivery delays are a key driver of negative reviews and churn
- **Who are the customers worth retaining?** — RFM segmentation identifies which customers to invest in vs which are already lost
- **How well do customers retain?** — Cohort analysis reveals whether Olist has a one-time buyer problem

---
## 3. Methodology

The Olist dataset is a **natural funnel dataset** — each order has explicit stage timestamps:

```
order_purchase_timestamp  →  order_approved_at
      →  order_delivered_carrier_date
      →  order_delivered_customer_date
      →  review_creation_date
```

**Data Model (6 tables joined):**

| Table | Rows | Role |
|---|---|---|
| `olist_orders` | 99,441 | Funnel backbone — timestamps per stage |
| `olist_order_items` | 112,650 | Revenue & product detail |
| `olist_order_payments` | 103,886 | Payment method & value |
| `olist_order_reviews` | 100,000 | Satisfaction score & timing |
| `olist_products` | 32,951 | Product categories |
| `olist_customers` | 99,441 | Customer identity & location |

---
## 4. Skills & Technologies

| Category | Tools |
|---|---|
| **Language** | Python 3.x |
| **Data Wrangling** | `pandas` — multi-table merges, datetime parsing, groupby, period indexing |
| **Visualisation** | `matplotlib`, `seaborn` — funnel charts, heatmaps, histograms, pie charts |
| **Funnel Analysis** | Stage engineering from timestamps, step conversion rates, drop-off quantification |
| **Cohort Analysis** | Monthly acquisition cohorts, retention matrix, avg retention curve |
| **RFM Segmentation** | Recency/Frequency/Monetary quintile scoring, segment labelling |
| **Delivery Analytics** | SLA analysis, on-time rate, delivery time distribution |
| **Business Intelligence** | Revenue waterfall, cancellation analysis, payment mix |

---
## 5. Data Loading & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('olist_outputs', exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.facecolor': '#f8fafc', 'figure.facecolor': 'white',
})
BLUE='#1565C0'; LBLUE='#42A5F5'; CYAN='#00BCD4'; GREEN='#43A047'
RED='#E53935'; AMBER='#FFB300'; NAVY='#0D47A1'; TEAL='#00897B'
OUT = 'olist_outputs/'
print('✅ Libraries loaded')

In [ ]:
DATA = 'olist_data/'
orders   = pd.read_csv(f'{DATA}olist_orders_dataset.csv',
    parse_dates=['order_purchase_timestamp','order_approved_at',
                 'order_delivered_carrier_date','order_delivered_customer_date',
                 'order_estimated_delivery_date'])
items    = pd.read_csv(f'{DATA}olist_order_items_dataset.csv')
payments = pd.read_csv(f'{DATA}olist_order_payments_dataset.csv')
reviews  = pd.read_csv(f'{DATA}olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date','review_answer_timestamp'])
products = pd.read_csv(f'{DATA}olist_products_dataset.csv')
customers= pd.read_csv(f'{DATA}olist_customers_dataset.csv')

cat_map = {
    'cama_mesa_banho':'Bed/Bath/Table','esporte_lazer':'Sports & Leisure',
    'moveis_decoracao':'Furniture & Decor','beleza_saude':'Beauty & Health',
    'utilidades_domesticas':'Home Utilities','automotivo':'Automotive',
    'informatica_acessorios':'IT Accessories','brinquedos':'Toys',
    'relogios_presentes':'Watches & Gifts','telefonia':'Telephony',
}
products['category_en'] = products['product_category_name'].map(cat_map).fillna(
    products['product_category_name'].str.replace('_',' ').str.title())

print(f'Orders    : {len(orders):,}')
print(f'Items     : {len(items):,}')
print(f'Payments  : {len(payments):,}')
print(f'Reviews   : {len(reviews):,}')
print(f'Customers : {customers["customer_unique_id"].nunique():,} unique')
print(f'Date range: {orders.order_purchase_timestamp.min().date()} → {orders.order_purchase_timestamp.max().date()}')
print(f'\nOrder status breakdown:')
print(orders["order_status"].value_counts())

---
## 6. Funnel 1 — Order Lifecycle

Traces every order from placement through approval, shipping, delivery, review, and 5-star rating.
This is the **core funnel** — it shows where volume leaks at each stage.

In [ ]:
total_orders       = len(orders)
approved           = orders['order_approved_at'].notna().sum()
shipped_to_carrier = orders['order_delivered_carrier_date'].notna().sum()
delivered          = (orders['order_status'] == 'delivered').sum()
reviewed           = reviews['order_id'].isin(orders['order_id']).sum()
five_star          = reviews[reviews['review_score']==5]['order_id'].nunique()

funnel1 = pd.DataFrame({
    'Stage': ['Order Placed','Payment Approved','Shipped to Carrier',
              'Delivered to Customer','Customer Reviewed','5-Star Review'],
    'Count': [total_orders, approved, shipped_to_carrier, delivered, reviewed, five_star],
    'Color': [BLUE, LBLUE, CYAN, TEAL, GREEN, AMBER]
})
funnel1['Overall%'] = funnel1['Count'] / funnel1['Count'].iloc[0] * 100
funnel1['Step%']    = funnel1['Count'] / funnel1['Count'].shift(1).fillna(funnel1['Count'].iloc[0]) * 100
funnel1['Dropoff']  = funnel1['Count'].shift(1).fillna(funnel1['Count'].iloc[0]) - funnel1['Count']

print(funnel1[['Stage','Count','Overall%','Step%','Dropoff']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
ax = axes[0]
max_w = funnel1['Count'].max()
for i, (_, row) in enumerate(funnel1.iterrows()):
    y   = len(funnel1)-1-i
    w   = row['Count'] / max_w
    off = (1-w) / 2
    ax.barh(y, w, left=off, height=0.70, color=row['Color'], alpha=0.92, zorder=3)
    ax.text(0.5, y+0.21, row['Stage'], ha='center', va='center', fontsize=9.5, fontweight='700', color='white', zorder=4)
    ax.text(0.5, y-0.19, f"{row['Count']:,}   ({row['Overall%']:.1f}%)", ha='center', va='center', fontsize=8.5, color='white', zorder=4, alpha=0.9)
    if i > 0:
        ax.text(off+w+0.01, y, f"-{100-row['Step%']:.1f}%", va='center', fontsize=8, color=RED, fontweight='600')
ax.set_xlim(0, 1.18); ax.set_yticks([]); ax.set_xticks([])
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_facecolor('#f0f4f8')
ax.set_title('Olist Order Lifecycle Funnel', fontsize=13, fontweight='bold', pad=14)

ax2 = axes[1]
step_convs  = funnel1['Step%'].iloc[1:].values
bar_cols    = [GREEN if v>=90 else AMBER if v>=70 else RED for v in step_convs]
bars = ax2.bar(range(len(funnel1)-1), step_convs, color=bar_cols, width=0.6, zorder=3, alpha=0.9)
ax2.axhline(100, color='#ddd', linewidth=1, linestyle='--')
ax2.set_xticks(range(len(funnel1)-1))
ax2.set_xticklabels(funnel1['Stage'].iloc[1:], rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('Step Conversion Rate (%)'); ax2.set_ylim(0, 115)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))
ax2.set_title('Stage-to-Stage Conversion Rates', fontsize=13, fontweight='bold', pad=14)
for bar, val in zip(bars, step_convs):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='700')
plt.tight_layout(); plt.savefig(f'{OUT}funnel1_order_lifecycle.png', bbox_inches='tight'); plt.show()

---
## 7. Funnel 2 — Revenue Conversion

Tracks R$ value through each order stage — revealing the true revenue waterfall from gross billing to payment collected.

In [ ]:
order_rev = items.groupby('order_id').agg(
    item_revenue=('price','sum'), freight=('freight_value','sum')).reset_index()
order_rev['freight_revenue'] = order_rev['freight']
order_rev['total_revenue']   = order_rev['item_revenue'] + order_rev['freight_revenue']
pay_per_order = payments.groupby('order_id')['payment_value'].sum().reset_index()
orders_full   = orders.merge(order_rev[['order_id','item_revenue','total_revenue']], on='order_id', how='left')
orders_full   = orders_full.merge(pay_per_order, on='order_id', how='left')

gross_rev         = orders_full['item_revenue'].sum()
approved_rev      = orders_full[orders_full['order_approved_at'].notna()]['item_revenue'].sum()
shipped_rev       = orders_full[orders_full['order_delivered_carrier_date'].notna()]['item_revenue'].sum()
delivered_rev     = orders_full[orders_full['order_status']=='delivered']['item_revenue'].sum()
cancelled_rev     = orders_full[orders_full['order_status']=='canceled']['item_revenue'].sum()
payment_collected = orders_full[orders_full['payment_value'].notna()]['payment_value'].sum()

print(f'Gross Revenue    : R${gross_rev:,.2f}')
print(f'Delivered Revenue: R${delivered_rev:,.2f}  ({delivered_rev/gross_rev*100:.1f}% of gross)')
print(f'Cancelled Revenue: R${cancelled_rev:,.2f}  ({cancelled_rev/gross_rev*100:.1f}% of gross)')
print(f'Payment Collected: R${payment_collected:,.2f}')

funnel2 = pd.DataFrame({
    'Stage'  : ['Gross Order Value','Payment Approved','Handed to Carrier','Successfully Delivered','Payment Collected'],
    'Revenue': [gross_rev, approved_rev, shipped_rev, delivered_rev, payment_collected],
    'Color'  : [BLUE, LBLUE, CYAN, GREEN, AMBER]
})
funnel2['Overall%'] = funnel2['Revenue'] / funnel2['Revenue'].iloc[0] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]; max_r = funnel2['Revenue'].max()
for i, (_, row) in enumerate(funnel2.iterrows()):
    y=len(funnel2)-1-i; w=row['Revenue']/max_r; off=(1-w)/2
    ax.barh(y, w, left=off, height=0.65, color=row['Color'], alpha=0.92, zorder=3)
    ax.text(0.5, y+0.18, row['Stage'], ha='center', va='center', fontsize=9.5, fontweight='700', color='white', zorder=4)
    ax.text(0.5, y-0.18, f"R${row['Revenue']:,.0f}  ({row['Overall%']:.1f}%)", ha='center', va='center', fontsize=8.5, color='white', zorder=4, alpha=0.9)
ax.set_xlim(0,1); ax.set_yticks([]); ax.set_xticks([])
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_facecolor('#f0f4f8'); ax.set_title('Revenue Conversion Funnel (R$)', fontsize=13, fontweight='bold', pad=14)

ax2 = axes[1]
pay_summary = payments.groupby('payment_type')['payment_value'].sum().sort_values(ascending=False)
pay_labels  = {'credit_card':'Credit Card','boleto':'Boleto','voucher':'Voucher','debit_card':'Debit Card','not_defined':'Other'}
pay_summary.index = [pay_labels.get(x, x) for x in pay_summary.index]
pay_pct = pay_summary / pay_summary.sum() * 100
ax2.pie(pay_pct.values, labels=pay_summary.index, autopct='%1.1f%%',
    colors=[BLUE,LBLUE,CYAN,GREEN,AMBER], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, pctdistance=0.72)
ax2.set_title('Payment Method Mix', fontsize=13, fontweight='bold', pad=14)
ax2.text(0,-1.45,f"Cancelled: R${cancelled_rev:,.0f} ({cancelled_rev/gross_rev*100:.1f}% of gross)",
    ha='center', fontsize=9, color=RED, fontweight='600')
plt.tight_layout(); plt.savefig(f'{OUT}funnel2_revenue_conversion.png', bbox_inches='tight'); plt.show()

---
## 8. Funnel 3 — Delivery Performance

Breaks down delivery time by stage and measures the on-time delivery rate against Olist's estimated delivery dates.

In [ ]:
delivered_orders = orders[orders['order_status']=='delivered'].copy()
delivered_orders['hrs_to_approve']   = (delivered_orders['order_approved_at'] - delivered_orders['order_purchase_timestamp']).dt.total_seconds()/3600
delivered_orders['days_to_carrier']  = (delivered_orders['order_delivered_carrier_date'] - delivered_orders['order_approved_at']).dt.total_seconds()/86400
delivered_orders['days_to_customer'] = (delivered_orders['order_delivered_customer_date'] - delivered_orders['order_delivered_carrier_date']).dt.total_seconds()/86400
delivered_orders['days_total']       = (delivered_orders['order_delivered_customer_date'] - delivered_orders['order_purchase_timestamp']).dt.total_seconds()/86400
delivered_orders['on_time']          = delivered_orders['order_delivered_customer_date'] <= delivered_orders['order_estimated_delivery_date']

on_time_pct  = delivered_orders['on_time'].mean() * 100
avg_total    = delivered_orders['days_total'].median()

print(f'On-Time Rate    : {on_time_pct:.1f}%')
print(f'Median Total    : {avg_total:.0f} days')
print(f'Median Approval : {delivered_orders["hrs_to_approve"].median():.1f} hours')
print(f'Median To Carrier: {delivered_orders["days_to_carrier"].median():.1f} days')
print(f'Median To Customer:{delivered_orders["days_to_customer"].median():.1f} days')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ax = axes[0]
stages  = ['Approval\n(days)', 'To Carrier\n(days)', 'To Customer\n(days)']
medians = [delivered_orders['hrs_to_approve'].median()/24, delivered_orders['days_to_carrier'].median(), delivered_orders['days_to_customer'].median()]
bars = ax.bar(stages, medians, color=[LBLUE,CYAN,TEAL], width=0.55, zorder=3, alpha=0.9)
ax.set_title('Median Time per Delivery Stage', fontsize=12, fontweight='bold'); ax.set_ylabel('Days')
for bar, val in zip(bars, medians): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.08, f'{val:.1f}d', ha='center', va='bottom', fontsize=12, fontweight='700')

ax2 = axes[1]
ax2.pie([on_time_pct, 100-on_time_pct], labels=['On-Time','Late'],
    autopct='%1.1f%%', colors=[GREEN,RED], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':3}, pctdistance=0.65, radius=0.85)
ax2.set_title(f'On-Time Delivery Rate\n(Median: {avg_total:.0f} days end-to-end)', fontsize=12, fontweight='bold')

ax3 = axes[2]
delivery_days = delivered_orders['days_total'].clip(0,60).dropna()
ax3.hist(delivery_days, bins=40, color=BLUE, alpha=0.8, edgecolor='white', linewidth=0.5, zorder=3)
ax3.axvline(delivery_days.median(), color=AMBER, linewidth=2.5, linestyle='--', label=f'Median: {delivery_days.median():.0f}d')
ax3.axvline(delivery_days.mean(),   color=RED,   linewidth=2.5, linestyle=':',  label=f'Mean: {delivery_days.mean():.0f}d')
ax3.set_title('Total Delivery Time Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Days from Purchase to Delivery'); ax3.set_ylabel('Orders'); ax3.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT}funnel3_delivery_performance.png', bbox_inches='tight'); plt.show()

---
## 9. Funnel 4 — Customer Satisfaction

Of all delivered orders, how many customers leave reviews — and what do they say?

In [ ]:
delivered = (orders['order_status'] == 'delivered').sum()
orders_with_review = reviews['order_id'].isin(orders['order_id']).sum()
positive_reviews   = reviews[reviews['review_score']>=4]['order_id'].nunique()
five_star_reviews  = reviews[reviews['review_score']==5]['order_id'].nunique()

sat_funnel = pd.DataFrame({
    'Stage': ['Delivered Orders','Left a Review','Positive (4-5★)','5-Star Review'],
    'Count': [delivered, orders_with_review, positive_reviews, five_star_reviews],
    'Color': [BLUE, CYAN, GREEN, AMBER]
})
sat_funnel['Overall%'] = sat_funnel['Count'] / sat_funnel['Count'].iloc[0] * 100
sat_funnel['Step%']    = sat_funnel['Count'] / sat_funnel['Count'].shift(1).fillna(sat_funnel['Count'].iloc[0]) * 100
print(sat_funnel[['Stage','Count','Overall%','Step%']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]; max_s = sat_funnel['Count'].max()
for i, (_, row) in enumerate(sat_funnel.iterrows()):
    y=len(sat_funnel)-1-i; w=row['Count']/max_s; off=(1-w)/2
    ax.barh(y, w, left=off, height=0.65, color=row['Color'], alpha=0.9, zorder=3)
    ax.text(0.5, y+0.18, row['Stage'], ha='center', va='center', fontsize=10, fontweight='700', color='white', zorder=4)
    ax.text(0.5, y-0.18, f"{row['Count']:,}   ({row['Overall%']:.1f}%)", ha='center', va='center', fontsize=8.5, color='white', zorder=4, alpha=0.9)
    if i>0: ax.text(off+w+0.01, y, f"-{100-row['Step%']:.1f}%", va='center', fontsize=8, color=RED, fontweight='600')
ax.set_xlim(0,1.15); ax.set_yticks([]); ax.set_xticks([])
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_facecolor('#f0f4f8'); ax.set_title('Customer Satisfaction Funnel', fontsize=13, fontweight='bold', pad=14)

ax2 = axes[1]
score_counts = reviews['review_score'].value_counts().sort_index()
bars = ax2.bar(score_counts.index, score_counts.values, color=[RED,'#FF7043',AMBER,LBLUE,GREEN], width=0.65, zorder=3, alpha=0.9)
ax2.set_title('Review Score Distribution', fontsize=13, fontweight='bold', pad=14)
ax2.set_xlabel('Review Score'); ax2.set_ylabel('Reviews'); ax2.set_xticks([1,2,3,4,5])
ax2.set_xticklabels(['1★','2★','3★','4★','5★'], fontsize=11)
for bar, val in zip(bars, score_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300, f'{val:,}\n({val/score_counts.sum()*100:.1f}%)', ha='center', va='bottom', fontsize=8.5, fontweight='600')
plt.tight_layout(); plt.savefig(f'{OUT}funnel4_satisfaction.png', bbox_inches='tight'); plt.show()

---
## 10. Cohort Retention

Groups customers by their first purchase month and tracks how many return in subsequent months.

In [ ]:
cust_orders = orders.merge(customers, on='customer_id', how='left')
cust_orders['purchase_month'] = cust_orders['order_purchase_timestamp'].dt.to_period('M')
cust_orders['cohort_month']   = cust_orders.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min').dt.to_period('M')
cust_orders['period_index']   = (cust_orders['purchase_month'] - cust_orders['cohort_month']).apply(lambda x: x.n)
cohort_pivot = cust_orders.groupby(['cohort_month','period_index'])['customer_unique_id'].nunique().reset_index().pivot(index='cohort_month', columns='period_index', values='customer_unique_id')
retention    = cohort_pivot.divide(cohort_pivot[0], axis=0) * 100
ret_trimmed  = retention.iloc[2:-3, :13].fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
mask = ret_trimmed == 0
sns.heatmap(ret_trimmed, annot=True, fmt='.0f', cmap='Blues',
    linewidths=0.5, linecolor='white', cbar_kws={'label':'Retention %','shrink':0.8},
    ax=axes[0], mask=mask, annot_kws={'size':7.5})
axes[0].set_title('Customer Cohort Retention Matrix', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Months Since First Purchase'); axes[0].set_ylabel('Cohort')
axes[0].set_yticklabels([str(p) for p in ret_trimmed.index], rotation=0, fontsize=8)

avg_ret = retention.iloc[2:-3, 1:13].mean()
axes[1].fill_between(avg_ret.index, avg_ret.values, alpha=0.2, color=BLUE)
axes[1].plot(avg_ret.index, avg_ret.values, marker='o', color=BLUE, linewidth=2.5, markersize=7, markerfacecolor='white', markeredgewidth=2.5)
for x, y in zip(avg_ret.index, avg_ret.values):
    axes[1].annotate(f'{y:.1f}%', (x,y), textcoords='offset points', xytext=(0,9), ha='center', fontsize=8.5, color=NAVY, fontweight='600')
axes[1].set_title('Avg Retention Rate by Month', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Months Since First Purchase'); axes[1].set_ylabel('Retention (%)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}%'))
axes[1].set_xticks(avg_ret.index); axes[1].set_ylim(0, max(avg_ret.values)*1.35)
plt.tight_layout(); plt.savefig(f'{OUT}cohort_retention.png', bbox_inches='tight'); plt.show()

---
## 11. Drop-off Deep Dive

Four supporting charts: order volume trend, top categories by revenue, monthly cancellation rate, and credit card installment risk distribution.

In [ ]:
orders_items = orders.merge(items, on='order_id', how='left').merge(products[['product_id','category_en']], on='product_id', how='left')
monthly_orders = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size().reset_index()
monthly_orders.columns = ['Month','Orders']; monthly_orders['Month_str'] = monthly_orders['Month'].astype(str)
cat_rev = orders_items.groupby('category_en')['price'].sum().sort_values(ascending=False).head(10)
cancel_monthly = orders[orders['order_status']=='canceled'].groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
total_monthly  = orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size()
cancel_rate = (cancel_monthly/total_monthly*100).fillna(0).reset_index(); cancel_rate.columns=['Month','Rate']; cancel_rate['Month_str']=cancel_rate['Month'].astype(str)
install_dist = payments[payments['payment_type']=='credit_card']['payment_installments'].value_counts().sort_index().head(12)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
ax=axes[0][0]
ax.fill_between(range(len(monthly_orders)), monthly_orders['Orders'], alpha=0.2, color=BLUE)
ax.plot(range(len(monthly_orders)), monthly_orders['Orders'], marker='o', color=BLUE, linewidth=2.5, markersize=4, markerfacecolor='white', markeredgewidth=2)
ax.set_title('Monthly Order Volume Trend', fontsize=12, fontweight='bold'); ax.set_ylabel('Orders')
step=max(1,len(monthly_orders)//10); ax.set_xticks(range(0,len(monthly_orders),step))
ax.set_xticklabels(monthly_orders['Month_str'].iloc[::step], rotation=35, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
peak_idx=monthly_orders['Orders'].idxmax()
ax.annotate(f"Peak\n{monthly_orders.loc[peak_idx,'Orders']:,}", xy=(peak_idx,monthly_orders.loc[peak_idx,'Orders']),
    xytext=(peak_idx-4,monthly_orders.loc[peak_idx,'Orders']+600), fontsize=8.5, color=AMBER, fontweight='700',
    arrowprops=dict(arrowstyle='->',color=AMBER,lw=1.5))

ax2=axes[0][1]
bar_c=[BLUE if i==0 else LBLUE if i<3 else CYAN if i<6 else '#90CAF9' for i in range(len(cat_rev))]
ax2.barh(cat_rev.index[::-1], cat_rev.values[::-1], color=bar_c[::-1], alpha=0.9, zorder=3)
ax2.set_title('Top 10 Product Categories by Revenue', fontsize=12, fontweight='bold'); ax2.set_xlabel('Revenue (R$)')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
for bar, val in zip(ax2.patches, cat_rev.values[::-1]):
    ax2.text(bar.get_width()+5000, bar.get_y()+bar.get_height()/2, f'R${val/1e6:.2f}M', va='center', fontsize=8, fontweight='600')

ax3=axes[1][0]
ax3.bar(range(len(cancel_rate)), cancel_rate['Rate'], color=RED, alpha=0.72, zorder=3)
ax3.axhline(cancel_rate['Rate'].mean(), color=AMBER, linewidth=2, linestyle='--', label=f"Avg: {cancel_rate['Rate'].mean():.2f}%")
ax3.set_title('Order Cancellation Rate by Month', fontsize=12, fontweight='bold'); ax3.set_ylabel('Cancellation Rate (%)')
step2=max(1,len(cancel_rate)//10); ax3.set_xticks(range(0,len(cancel_rate),step2))
ax3.set_xticklabels(cancel_rate['Month_str'].iloc[::step2], rotation=35, ha='right', fontsize=8); ax3.legend(fontsize=9)

ax4=axes[1][1]
ax4.bar(install_dist.index, install_dist.values,
    color=[BLUE if i<=3 else LBLUE if i<=6 else CYAN for i in install_dist.index], width=0.7, zorder=3, alpha=0.9)
ax4.set_title('Credit Card Installments Distribution\n(Higher installments = higher payment risk)', fontsize=11, fontweight='bold')
ax4.set_xlabel('Installments'); ax4.set_ylabel('Transactions'); ax4.set_xticks(install_dist.index)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout(); plt.savefig(f'{OUT}dropoff_deep_dive.png', bbox_inches='tight'); plt.show()

---
## 12. RFM Segmentation

Scores every customer on Recency, Frequency, and Monetary value (1–5 quintiles each) and maps them to six actionable segments.

In [ ]:
cust_rev = items.merge(orders[['order_id','customer_id','order_purchase_timestamp','order_status']], on='order_id')
cust_rev = cust_rev[cust_rev['order_status']=='delivered'].merge(customers[['customer_id','customer_unique_id']], on='customer_id')
cust_rev['revenue'] = cust_rev['price'] + cust_rev['freight_value']
SNAPSHOT = cust_rev['order_purchase_timestamp'].max()
rfm = cust_rev.groupby('customer_unique_id').agg(
    Recency=('order_purchase_timestamp', lambda x: (SNAPSHOT-x.max()).days),
    Frequency=('order_id','nunique'), Monetary=('revenue','sum')).reset_index()
rfm['R'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM'] = rfm['R'] + rfm['F'] + rfm['M']
def segment(row):
    if row['RFM']>=13: return 'Champions'
    elif row['RFM']>=10: return 'Loyal'
    elif row['RFM']>=8: return 'Potential Loyalists'
    elif row['R']>=4: return 'Recent'
    elif row['RFM']>=6: return 'At Risk'
    else: return 'Lost'
rfm['Segment'] = rfm.apply(segment, axis=1)
seg_colors_map = {'Champions':BLUE,'Loyal':LBLUE,'Potential Loyalists':CYAN,'Recent':GREEN,'At Risk':AMBER,'Lost':RED}

print(rfm.groupby('Segment').agg(Count=('customer_unique_id','count'),Avg_Recency=('Recency','mean'),Avg_Orders=('Frequency','mean'),Avg_Revenue=('Monetary','mean')).round(1).sort_values('Avg_Revenue', ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
seg_counts=rfm['Segment'].value_counts(); bc=[seg_colors_map.get(s,'#999') for s in seg_counts.index]
bars=axes[0].bar(seg_counts.index, seg_counts.values, color=bc, width=0.6, zorder=3, alpha=0.9)
axes[0].set_title('Customers by RFM Segment', fontsize=12, fontweight='bold'); axes[0].set_ylabel('Customers')
axes[0].set_xticklabels(seg_counts.index, rotation=20, ha='right')
for bar, val in zip(bars, seg_counts.values): axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+150, f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='700')

seg_rev=rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False); bc2=[seg_colors_map.get(s,'#999') for s in seg_rev.index]
bars2=axes[1].barh(seg_rev.index[::-1], seg_rev.values[::-1], color=bc2[::-1], zorder=3, alpha=0.9)
axes[1].set_title('Total Revenue by Segment', fontsize=12, fontweight='bold'); axes[1].set_xlabel('Revenue (R$)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1e6:.1f}M'))
for bar, val in zip(bars2, seg_rev.values[::-1]): axes[1].text(bar.get_width()+5000, bar.get_y()+bar.get_height()/2, f'R${val/1e6:.2f}M', va='center', fontsize=9, fontweight='600')

seg_avg=rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False); bc3=[seg_colors_map.get(s,'#999') for s in seg_avg.index]
bars3=axes[2].barh(seg_avg.index[::-1], seg_avg.values[::-1], color=bc3[::-1], zorder=3, alpha=0.9)
axes[2].set_title('Avg Revenue per Customer', fontsize=12, fontweight='bold'); axes[2].set_xlabel('Avg Revenue (R$)')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x:,.0f}'))
for bar, val in zip(bars3, seg_avg.values[::-1]): axes[2].text(bar.get_width()+20, bar.get_y()+bar.get_height()/2, f'R${val:,.0f}', va='center', fontsize=9, fontweight='600')
plt.tight_layout(); plt.savefig(f'{OUT}rfm_segmentation.png', bbox_inches='tight'); plt.show()

---
## 13. Results & Business Recommendations

### Funnel 1 — Order Lifecycle
Olist has a **strong fulfilment rate (97%)** — most orders are delivered. The biggest drop is at the **review stage**: not all customers leave feedback, and only 57% give 5 stars. The journey from delivered → positive review is the biggest satisfaction gap.

> **💡 Recommendation:** Send a post-delivery review nudge at Day 3 and Day 7. Personalise the request with the product name. Offer a small loyalty credit for completing a review.

---

### Funnel 2 — Revenue Conversion
Gross revenue converts very efficiently to collected payments — the marketplace mechanics are sound. Cancellations are **low at 0.6%** but worth monitoring during high-volume periods.

> **💡 Recommendation:** Investigate cancellations by category and seller — are specific sellers driving disproportionate cancellations? Monitor credit card instalment defaults (10–12 instalments) as a forward-looking bad debt signal.

---

### Funnel 3 — Delivery Performance
**91.9% on-time** is a strong result. Median delivery is 10 days. The approval-to-carrier handoff (seller processing time) is the biggest controllable variable.

> **💡 Recommendation:** Flag sellers whose median handling time exceeds 3 days. Introduce a seller performance score tied to their handling speed. Offer premium placement to sellers who consistently ship within 24 hours.

---

### Funnel 4 — Satisfaction
**76.3% positive reviews** (4–5 stars) is solid. 1-star reviews (11.8%) are the highest-risk signal — these customers are lost and will likely not return.

> **💡 Recommendation:** Trigger an immediate customer service outreach for any 1-star review. Build a recovery playbook: refund offer, replacement, or escalation to the seller within 24 hours.

---

### Cohort Retention
Olist has a **classic single-purchase problem** — the vast majority of customers buy once and never return. This is typical for marketplace models but represents a significant CLV opportunity.

> **💡 Recommendation:** Launch a post-purchase loyalty programme. Send "What to buy next" recommendations based on purchased category 14 days after delivery. Offer a second-order discount to all first-time buyers.

---

### RFM: At Risk + Lost = 23,560 customers
Nearly a quarter of the customer base is at risk of permanent churn or already lost.

> **💡 Recommendation:** Champions → VIP early access; Loyal → referral programme; At Risk → urgent win-back with time-limited offer; Lost → lightweight re-engagement survey to understand departure reason.

---
## 14. Next Steps

| Priority | Action | Method |
|---|---|---|
| 🔴 High | **Churn prediction model** | XGBoost on RFM + recency trend + category → predict 30-day churn probability |
| 🔴 High | **Seller performance analysis** | Correlate seller handling time, cancellation rate → delivery performance and review score |
| 🟡 Medium | **Basket/cross-sell analysis** | Apriori association rules → recommend complementary categories |
| 🟡 Medium | **Geographic delivery analysis** | Map delivery performance by Brazilian state → identify underserved regions |
| 🟡 Medium | **CLV forecasting** | BG/NBD model to predict customer lifetime value per segment |
| 🟢 Low | **Tableau dashboard** | Migrate all 4 funnels + cohort + RFM into interactive Tableau |

---
*Data: Olist Brazilian E-Commerce Public Dataset. Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce*

In [ ]:
import os
os.makedirs('olist_outputs', exist_ok=True)

# Re-compute key metrics if running standalone
delivered = (orders['order_status'] == 'delivered').sum()
orders_with_review = reviews['order_id'].isin(orders['order_id']).sum()
positive_reviews   = reviews[reviews['review_score']>=4]['order_id'].nunique()
five_star_reviews  = reviews[reviews['review_score']==5]['order_id'].nunique()

order_rev = items.groupby('order_id').agg(item_revenue=('price','sum')).reset_index()
gross_rev = orders.merge(order_rev, on='order_id', how='left')['item_revenue'].sum()

summary = pd.DataFrame([
    ['Total Orders',                 f"{len(orders):,}"],
    ['Delivered Orders',             f"{delivered:,} ({delivered/len(orders)*100:.1f}%)"],
    ['Cancelled Orders',             f"{(orders['order_status']=='canceled').sum():,}"],
    ['On-Time Delivery Rate',        f"{on_time_pct:.1f}%"],
    ['Median Delivery Days',         f"{avg_total:.0f} days"],
    ['Review Rate',                  f"{orders_with_review/delivered*100:.1f}%"],
    ['Positive Review Rate (4-5★)',  f"{positive_reviews/orders_with_review*100:.1f}%"],
    ['5-Star Review Rate',           f"{five_star_reviews/orders_with_review*100:.1f}%"],
    ['Total Gross Revenue (R$)',     f"R${gross_rev:,.2f}"],
    ['Avg Order Value (R$)',         f"R${items.groupby('order_id')['price'].sum().mean():,.2f}"],
    ['Unique Customers',             f"{customers['customer_unique_id'].nunique():,}"],
    ['Product Categories',           f"{products['product_category_name'].nunique():,}"],
], columns=['Metric','Value'])
summary.to_csv('olist_outputs/olist_summary_stats.csv', index=False)
summary.style.set_properties(**{'text-align':'left'})